# Backblaze Drive Stats — download the 2024 dataset straight to Drive

Fetches the four 2024 quarterly zips from Backblaze's public bucket onto this Colab
runtime's **ephemeral disk**, then extracts ONLY the daily `YYYY-MM-DD.csv` files onto
Google Drive under `Data/Backblaze/` — no zip ever touches Drive and nothing goes
through your local machine. Stdlib-only: no pip installs, no GPU, no repo clone needed.

- **Space math (approximate):** a 2024 quarter zip is ~1.5–3 GB and extracts to roughly
  10 GB of daily CSVs; **the full year lands at ~40 GB on Drive**. Trim `QUARTERS` if
  space is tight — the campaign reads a scoped subset (`backblaze_models` / date range /
  SMART columns, CHANGES.md §56), but the raw day files must exist on Drive.
- **Restartable:** a quarter whose day files are all present is skipped outright; a
  partial quarter re-downloads its zip and fills only the missing/short files.
- The repo's loader finds day files by a **recursive glob at any nesting**
  (`src/datasets/backblaze.py`), so the zip's internal folder (e.g. `data_Q1_2024/`)
  is kept as-is — nothing needs renaming or flattening.
- If a URL 404s, Backblaze may have reorganized the bucket — check
  https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data and adjust
  `BASE_URL` / the zip name pattern below.


In [ ]:
# 1) mount Drive + choose what to fetch
from google.colab import drive
drive.mount('/content/drive')

import os

YEAR      = 2024
QUARTERS  = [1, 2, 3, 4]              # trim to save Drive space (each quarter ~10 GB)
DRIVE     = '/content/drive/MyDrive/pdm_tsfm'   # same Drive root as the campaign notebooks
DATA_ROOT = f'{DRIVE}/Data'
DEST      = f'{DATA_ROOT}/Backblaze'  # day files land under here (zip nesting kept)
BASE_URL  = 'https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data'
TMP       = '/content/backblaze_zips' # ephemeral — zips never touch Drive

os.makedirs(DEST, exist_ok=True)
os.makedirs(TMP, exist_ok=True)
print('day files will land under:', DEST)


In [ ]:
# 2) plan — which quarters are already complete on Drive, and how big the rest are
import calendar, datetime as dt, re, urllib.request

DAY_RE = re.compile(r'^\d{4}-\d{2}-\d{2}\.csv$')

def quarter_dates(year, q):
    days = []
    for m in range(3 * q - 2, 3 * q + 1):
        for d in range(1, calendar.monthrange(year, m)[1] + 1):
            days.append(dt.date(year, m, d))
    return days

def found_days():
    out = {}
    for root, _dirs, files in os.walk(DEST):
        for f in files:
            if DAY_RE.match(f):
                out.setdefault(f[:-4], []).append(os.path.join(root, f))
    return out

def remote_size(url):
    try:
        req = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(req, timeout=30) as r:
            return int(r.headers.get('Content-Length', 0))
    except Exception as e:
        print(f'  (could not HEAD {url}: {e})')
        return None

have = found_days()
plan = []
for q in QUARTERS:
    expected = quarter_dates(YEAR, q)
    missing = [d for d in expected if d.isoformat() not in have]
    url = f'{BASE_URL}/data_Q{q}_{YEAR}.zip'
    if not missing:
        print(f'Q{q} {YEAR}: complete ({len(expected)} days) — skipping')
        continue
    size = remote_size(url)
    print(f'Q{q} {YEAR}: {len(missing)}/{len(expected)} days missing'
          + (f'  (zip ~{size/1e9:.2f} GB)' if size else ''))
    plan.append((q, url))
print(f'\n{len(plan)} quarter(s) to fetch.')


In [ ]:
# 3) download each planned quarter to ephemeral disk, extract the day CSVs to Drive,
#    delete the local zip. Safe to interrupt and re-run — it resumes at the plan cell.
import shutil, zipfile

CHUNK = 1 << 22   # 4 MiB

def download(url, dest_zip):
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(url, timeout=60) as r, open(dest_zip, 'wb') as f:
                total = int(r.headers.get('Content-Length', 0))
                done = 0
                while True:
                    chunk = r.read(CHUNK)
                    if not chunk:
                        break
                    f.write(chunk)
                    done += len(chunk)
                    if done % (CHUNK * 64) < CHUNK:   # progress every ~256 MB
                        print(f'  … {done/1e9:.2f} / {total/1e9:.2f} GB', flush=True)
            return
        except Exception as e:
            print(f'  attempt {attempt}/3 failed: {e}')
            if attempt == 3:
                raise

def extract(zip_path):
    kept = skipped = 0
    dest_root = os.path.normpath(DEST)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            base = os.path.basename(m.filename)
            if m.is_dir() or '__MACOSX' in m.filename or not DAY_RE.match(base):
                continue                              # keep ONLY day-named CSVs
            out = os.path.normpath(os.path.join(DEST, m.filename))
            if not out.startswith(dest_root + os.sep):
                continue                              # zip-slip guard
            if os.path.exists(out) and os.path.getsize(out) == m.file_size:
                skipped += 1
                continue
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with z.open(m) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst, CHUNK)
            kept += 1
    print(f'  extracted {kept} day files ({skipped} already present)')

for q, url in plan:
    zip_path = os.path.join(TMP, f'data_Q{q}_{YEAR}.zip')
    print(f'\nQ{q} {YEAR} — downloading {url}')
    download(url, zip_path)
    print(f'Q{q} {YEAR} — extracting to {DEST}')
    extract(zip_path)
    os.remove(zip_path)   # free ephemeral disk before the next quarter
print('\ndone.')


In [ ]:
# 4) verify — per-quarter completeness + total size on Drive
have = found_days()
total_bytes = sum(os.path.getsize(p) for paths in have.values() for p in paths)
print(f'{len(have)} day files under {DEST}  ({total_bytes/1e9:.1f} GB)')
if have:
    print('date range:', min(have), '→', max(have))
for q in QUARTERS:
    expected = quarter_dates(YEAR, q)
    missing = [d.isoformat() for d in expected if d.isoformat() not in have]
    line = f'Q{q} {YEAR}: {len(expected) - len(missing)}/{len(expected)} days'
    if missing:
        line += f'  MISSING e.g. {missing[:5]}'
    print(line)
dupes = {d: paths for d, paths in have.items() if len(paths) > 1}
if dupes:
    print(f'WARNING: {len(dupes)} date(s) present more than once — was a quarter '
          f'extracted twice into different subfolders? e.g. {sorted(dupes)[0]}')
print('\nNext: run notebooks/campaign/milestone_3/<model>.ipynb — the campaign scopes '
      'Backblaze via campaign.DEFAULT_DATASET_OVERRIDES + config.backblaze_models.')
